# Stage 2 Lab 01: 工業級預訓練模型微創嫁接大對決 — `Qwen2.5-0.5B + 標準 LoRA` vs `Dual-Brain Qwen2.5-0.5B`

在 Stage 1 的白盒實驗（特に [Issue #4](https://github.com/EndeavorYen/LLM-learning/issues/4)）中，我們發現了**「寬語義骨幹（Wide Backbone）+ 窄意念瓶頸（Narrow Thought Funnel, 128D）」**的黃金法則。

本實驗直接挑戰終極目標：在只有一顆官方預訓練好的 **`Qwen/Qwen2.5-0.5B`（4.94 億參數、24 層、896 維隱藏空間、18 兆字元世界知識）** 的前提下，比較兩種改裝路線：
1. **對手 A（工業界標準路線：`Qwen2.5-0.5B + Standard Causal LoRA SFT`）**：維持傳統「由左到右逐字猜測」的自迴歸架構，掛載 LoRA 便利貼進行監督微調。
2. **對手 B（我們的微創雙腦嫁接路線：`Dual-Brain Grafted Qwen2.5-0.5B`）**：
   - **第一刀（100% 凍結 4.94 億預訓練腦）**：完整保留 Qwen2.5 讀過 18 兆字的世界常識與第 24 層 `896 維` 語義提取能力。
   - **第二刀（嫁接 `896 -> 128` 窄瓶頸正交意念漏斗）**：將 `896 維` 壓進 `128 維` 正交意念空間（`64 維領域座標 d` $\perp$ `64 維通用因果規律座標 s`），強迫濾除表面字詞雜訊。
   - **第三刀（快慢雙腦路由 + 未知領域謙遜閘門 OOD Humility Gate）**：熟練的電話與金鑰走 **系統一（快思肌肉記憶）** 平行顯影；沒見過的科幻/工程/醫學新系統（`MinConf < 85%`）自動切換至 **系統二（128 維意念跳躍腦）** 進行零樣本跨領域因果推演！


In [1]:
# Full Colab L4 Benchmark Script: Qwen/Qwen2.5-0.5B + LoRA vs Dual-Brain Grafted Qwen2.5-0.5B
import os, sys, time, math, random
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
qwen_base = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map="cuda")
qwen_base.eval()
for p in qwen_base.parameters():
    p.requires_grad = False  # 100% Freeze the 494M pre-trained weights!


[colab] Allocating L4 GPU session 'stage2_qwen_pk_v2'...
[colab] Session 'stage2_qwen_pk_v2' READY on gpu-l4-s-ass1b0-sarm6xjnz8mp
=== [1] Loading Official Pre-trained 'Qwen/Qwen2.5-0.5B' (494M Params, 24 Layers, 896-D Hidden State) ===
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")
Loaded in 23.8s! Hidden Dimension = 896, Layers = 24, Vocab = 151936
=== [2] Training Competitor A: Standard Qwen2.5-0.5B + LoRA SFT (1.38M LoRA params) ===
  [Competitor A: Qwen2.5-0.5B + LoRA SFT] Step   1/120 | SFT Loss: 4.9916
  [Competitor A: Qwen2.5-0.5B + LoRA SFT] Step  60/120 | SFT Loss: 0.0608
  [Competitor A: Qwen2.5-0.5B + LoRA SFT] Step 120/120 | SFT Loss: 0.0597

=== [3] Training Competitor B: Dua